In [5]:
import pennylane as qml
import numpy as np

In [17]:
import torch

In [18]:
seed = 4321
np.random.seed(seed=seed)
torch.manual_seed(seed=seed)

In [8]:
from sklearn.datasets import load_breast_cancer

x,y = load_breast_cancer(return_X_y=True)

In [9]:
# vamos a entrenar al modelo con parada temprana por perdida de validacion
# creamos base de datos de entrenamiento, de prueba y de validacion
from sklearn.model_selection import train_test_split
x_tr, x_test, y_tr, y_test = train_test_split(x, y, train_size=0.8, shuffle=True)
x_val, x_test, y_val, y_test = train_test_split(x_test, y_test, train_size=0.5, shuffle=True)

In [10]:
# todas las variables en la base de datos son positivas pero no estan normalizadas
from sklearn.preprocessing import MaxAbsScaler
scaler = MaxAbsScaler()
x_tr = scaler.fit_transform(x_tr)

x_test = scaler.transform(x_test)
x_val = scaler.transform(x_val)

# restringir todos los valores entre 0 y 1
x_test = np.clip(x_test, 0, 1)
x_val = np.clip(x_val, 0, 1)

In [11]:
print(len(x_tr[0]))

30


La base de datos que vamos a emplear tiene 30 variables, y a dia de hoy no tenemos acceso a ordenadores con 30 qubits, por lo que hemos de considerar una de las siguientes opciones: 
1. Usar amplitude encoding como feature map sobre 5 qubits (2**5 = 32)
2. Usar cualquier otro feature map pero con una reduccion de la dimensionalidad

METODO 1: AMPLITUDE ENCODING

METODO 2: REDUCCION DE LA DIMENSIONALIDAD

Vamos a restringir el caso a un circuito de 4 qubits

In [12]:
from sklearn.decomposition import PCA

pca = PCA(n_components=4)
xs_tr = pca.fit_transform(x_tr)
xs_test = pca.transform(x_test)
xs_val = pca.transform(x_val)

In [13]:
# Vamos a usar ZZ feature map y la forma variacional two-local
from itertools import combinations

def ZZFeatureMap(nqubits, data):


    nload = min(len(data), nqubits)
    for i in range(nload):
        qml.Hadamard(i)
        qml.RZ(2 * data[i], wires = i)
    
    for pair in list(combinations(range(nload),2)):
        q0 = pair[0]
        q1 = pair[1]

        qml.CNOT(wires=[q0, q1])
        qml.RZ(2.0 * (np.pi-data[q1])*(np.pi-data[q0]), wires=q1)
        qml.CNOT(wires=[q0, q1])

def TwoLocal(nqubits, theta, reps=1):
    for r in range(reps):
        for j in range(nqubits):
            qml.RY(theta[r*nqubits+j], wires=j)
        
        for j in range(nqubits-1):
            qml.CNOT(wires=[j,j+1])
    
    for j in range(nqubits):
        qml.RY(theta[reps*nqubits+j], wires=j)


En lugar de pedir a pennylane que devuelva las probabilidades de medida, le pediremos que devuelva el valor esperado del operador:
$M =
\begin{pmatrix}
1 & 0\\
0 & 0
\end{pmatrix}$

In [14]:
state_0 = [[1],[0]]
M = state_0*np.conj(state_0).T

In [15]:
nqubits = 4
dev = qml.device('default.qubit', wires=nqubits)

def qnn_circuit(inputs, theta):
    ZZFeatureMap(nqubits=nqubits, data=inputs)
    TwoLocal(nqubits=nqubits, theta=theta, reps=1) # por simplicidad se pide una repeticion unicamente
    return(qml.expval(qml.Hermitian(M, wires=[0])))

# se añade el argumento interface = 'tf' al inicializar el nodo para que trabaje con tensorflow
# si hubieramos usado el decorador @qml.qnode, no podriamos indicarle que trabaje con tensorflow
qnn = qml.QNode(qnn_circuit, dev, interface='tf')

In [21]:
# Por la interoperabilidad de pennylane podremos usar tensorflow para entrenar la red neuronal cuantica
weights = {'theta': 8}
